# Las cabezas de una capa y las cabezas de inducción

Notebook opcional del **módulo 4** del curso (en la app: `uv run dev` y la ruta `#/modulo/multi-cabeza`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

In [ ]:
import matplotlib.pyplot as plt
import torch
from transformers import AutoModel, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
modelo = AutoModel.from_pretrained("gpt2", output_attentions=True, attn_implementation="eager")

Una secuencia de tokens aleatorios repetida dos veces. Una **cabeza de inducción** atiende, en la segunda mitad, al token que siguió a la aparición anterior del token actual.

In [ ]:
torch.manual_seed(0)
n = 40
mitad = torch.randint(1000, 20000, (1, n))
ids = torch.cat([mitad, mitad], dim=1)
with torch.no_grad():
    atts = modelo(ids).attentions                      # (capas) x (1, cabezas, 2n, 2n)
puntuacion = torch.zeros(len(atts), atts[0].shape[1])
for c, a in enumerate(atts):
    for i in range(n, 2 * n):
        puntuacion[c] += a[0, :, i, i - n + 1]          # peso hacia el token siguiente a la copia anterior
puntuacion /= n
plt.imshow(puntuacion.numpy(), cmap="magma")
plt.xlabel("cabeza")
plt.ylabel("capa")
plt.colorbar(label="puntuación de inducción")
plt.show()
print("mejor cabeza (capa, cabeza):", divmod(int(puntuacion.argmax()), puntuacion.shape[1]))